# Gaming Toxicity Detection

**Authors:** Beibarys Nyussupov, Ruby Ngo, Paola Calle, Jett Forward

In [3]:
# libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 

# reproducibility
seed = 7524
np.random.seed(seed)

# remove limit at number of columns and rows shown
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# project root (notebooks/gaming/ -> notebooks/ -> project root)
PROJECT_ROOT = Path().resolve().parent.parent

In [4]:
# data directories
DATA_DIR_WOT  = PROJECT_ROOT / "data/raw_data/world_of_tanks/"
DATA_DIR_DOTA = PROJECT_ROOT / "data/raw_data/dota/"

## World of Tanks

#### Inspect each split

In [5]:
# import data 
train = pd.read_csv(DATA_DIR_WOT / "train.csv")
val = pd.read_csv(DATA_DIR_WOT / "val.csv")

# test dataset 
test_text = pd.read_csv(DATA_DIR_WOT /  "test_index_text.csv")
test_label = pd.read_csv(DATA_DIR_WOT / "test_index_label.csv")

# check the data 
for name, df in [("train", train), ("val", val), ("test_text", test_text), ("test_label", test_label)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"First few rows of the dataset:\n{df.head(3)}")
    print()

--- train ---
Shape: (42959, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

--- val ---
Shape: (5367, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index       message  label
0  31697  e50 is there    0.0
1  52311   this scouts    0.0
2   2775       i guess    0.0

--- test_text ---
Shape: (5375, 2)
Columns: ['index', 'message']
First few rows of the dataset:
   index         message
0  17775  aim you monkey
1  28228   Pz kill artys
2  19888             ggs

--- test_label ---
Shape: (5375, 2)
Columns: ['index', 'label']
First few rows of the dataset:
   index  label
0  17775    2.0
1  28228    0.0
2  19888    0.0



The test set is stored in two separate files - text and labels indexed by the same `index` column. Train and val are already complete. We need to join the test files on `index` before merging everything.

#### Reconstruct test split and merge all

In [6]:
# merge test text and labels
test = test_text.merge(test_label, on="index", how="inner")

# concatenate all data
wot = pd.concat([train, val, test], ignore_index=True)

print(f"Test join shape: {test.shape}\n")
print(f"Merged shape: {wot.shape}\n")
print(f"First few rows of the data:\n{wot.head(3)}\n")
print(f"Information about the data: {wot.info()}")

Test join shape: (5375, 3)

Merged shape: (53701, 3)

First few rows of the data:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53701 entries, 0 to 53700
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   index    53701 non-null  int64  
 1   message  53701 non-null  object 
 2   label    53701 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ MB
Information about the data: None


The test join uses `inner` - any index present in only one file would be silently dropped. If test shape after join is smaller than either file's length, there is an alignment issue in the original data. 

#### Final verification

In [7]:
# basic data checks
print("=== Shape ===")
print(wot.shape)

print("\n=== Nulls ===")
print(wot.isnull().sum())

print("\n=== Duplicate messages ===")
print(wot.duplicated(subset="message").sum())

print("\n=== Overall label distribution ===")
counts = wot["label"].value_counts().sort_index()
pct = wot["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
print(pd.DataFrame({"count": counts, "pct%": pct}))

=== Shape ===
(53701, 3)

=== Nulls ===
index      0
message    0
label      0
dtype: int64

=== Duplicate messages ===
19088

=== Overall label distribution ===
       count  pct%
label             
0.0    43497  81.0
1.0     7407  13.8
2.0     2343   4.4
3.0      349   0.6
4.0       75   0.1
5.0       30   0.1


We do not have any null values, however we got 19k duplicate messages.

We will explore duplicate rows and other issues in EDA. For now, let's save merged dataset to a parquet. 



In [15]:
# save to parquet for easier loading in future steps
wot.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

## Dota

#### Inspect each split

In [9]:
# import data
train = pd.read_csv(DATA_DIR_DOTA / "CONDA_train.csv")
val = pd.read_csv(DATA_DIR_DOTA / "CONDA_valid.csv")

# inspect each split
for name, df in [("train", train), ("val", val)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}\n")
    print(f"Columns: {list(df.columns)}\n")
    print(f"First few rows of the data:\n{df.head(3)}\n")

--- train ---
Shape: (26921, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversationId utterance  chatTime  playerSlot  \
0  11263      697            3193      wow!        76           0   
1  13741      843            3809       WTF      1563           5   
2  22125     1412            6199   wpe wpe      2853           1   

                  playerId intentClass slotClasses          slotTokens  
0  ANTS IN MY EYES JOHNSON           O          O            wow (O),   
1                      M.k           O          T            WTF (T),   
2              Acqua Ragia           O        O O   wpe (O), wpe (O),   

--- val ---
Shape: (8974, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversa

#### Clean, merge and standardise

In [10]:
# map intentClass to numeric label consistent with WoT schema
label_map = {'O': 0, 'E': 1, 'A': 2, 'I': 3}

# add split column and map labels
train["split"] = "train"
val["split"]   = "val"

# merge train + val (no test split in CONDA)
dota = pd.concat([train, val], ignore_index=True)

# keep only relevant columns and rename to match WoT schema
dota = dota[["Id", "utterance", "intentClass", "split"]].rename(columns={
    "Id": "index",
    "utterance":   "message",
    "intentClass": "label",
})

# map string labels to numeric
dota["label"] = dota["label"].map(label_map)

# drop nulls in message (7 train + 1 val)
dota = dota.dropna(subset=["message"]).reset_index(drop=True)

# shape 
print(f"Merged shape: {dota.shape}\n")
print(f"First few rows of the data:\n{dota.head(3)}\n")
print("\nInformation about the dataset:\n")
dota.info()

Merged shape: (35887, 4)

First few rows of the data:
   index  message  label  split
0  11263     wow!      0  train
1  13741      WTF      0  train
2  22125  wpe wpe      0  train


Information about the dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   index    35887 non-null  int64 
 1   message  35887 non-null  object
 2   label    35887 non-null  int64 
 3   split    35887 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


Only `utterance`, `intentClass`, `Id`, and `split` are kept = the remaining columns (matchId, conversationId, chatTime, playerSlot, playerId, slotClasses, slotTokens) carry no linguistic content relevant to toxicity detection. Labels are mapped to integers matching the WoT schema where 0 = non-toxic. Null utterances are dropped - they carry no text signal.

#### Final verification

In [11]:
print("=== Shape ===")
print(dota.shape)

print("\n=== Nulls ===")
print(dota.isnull().sum())

print("\n=== Duplicate messages ===")
print(dota.duplicated(subset="message").sum())

print("\n=== Label distribution ===")
counts = dota["label"].value_counts().sort_index()
pct    = dota["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
label_names = {0: "Other (O)", 1: "Ego (E)", 2: "Aggression (A)", 3: "Impolite (I)"}
print(pd.DataFrame({"count": counts, "pct%": pct}).rename(index=label_names))

=== Shape ===
(35887, 4)

=== Nulls ===
index      0
message    0
label      0
split      0
dtype: int64

=== Duplicate messages ===
11072

=== Label distribution ===
                count  pct%
label                      
Other (O)       26603  74.1
Ego (E)          4711  13.1
Aggression (A)   2299   6.4
Impolite (I)     2274   6.3


No null values after dropping 8 empty utterances. Duplicates exist but will be explored in EDA - short gaming phrases like "gg" appear frequently across matches and are expected repeats, not data errors. Label distribution mirrors WoT: heavy skew toward non-toxic (O), with Ego, Aggression, and Impolite as minority classes. Save to parquet.

In [14]:
# save to parquet for easier loading in future steps
dota.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)